In [1]:


import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# set or create an experiment
mlflow.set_experiment("exp_5 ml_algo_with_hp_tunning")


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/7', creation_time=1764058549473, experiment_id='7', last_update_time=1764058549473, lifecycle_stage='active', name='exp_5 ml_algo_with_hp_tunning', tags={}>

In [3]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [4]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [5]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import mlflow
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns


# -------------------------
# Target Replace
# -------------------------
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})


# -------------------------
# Split Numeric + Text
# -------------------------
X_numeric = df.iloc[:, 1:-1]
y = df['sentiment_numeric']

scaler = StandardScaler(with_mean=False)
X_numeric_scaled = scaler.fit_transform(X_numeric)

X_train_num, X_test_num, y_train, y_test, train_idx, test_idx = train_test_split(
    X_numeric_scaled, y, df.index,
    test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(ngram_range=(1,3), max_features=20000)
X_train_text = tfidf.fit_transform(df.loc[train_idx, 'text_clean'])
X_test_text = tfidf.transform(df.loc[test_idx, 'text_clean'])

X_train = sp.hstack([X_train_text, sp.csr_matrix(X_train_num)])
X_test = sp.hstack([X_test_text, sp.csr_matrix(X_test_num)])


# -------------------------
# Class Weights
# -------------------------
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weight_dict = {c: w for c, w in zip(np.unique(y_train), class_weights)}
print("Class Weights:", weight_dict)


# -------------------------
# Optuna Objective
# -------------------------
def objective(trial):

    C = trial.suggest_float("C", 0.01, 5.0, log=True)
    loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])

    model = LinearSVC(
        C=C,
        loss=loss,
        class_weight=weight_dict,
        max_iter=3000
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return recall_score(y_test, preds, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("\nBest Hyperparameters:", study.best_params)

best_params = study.best_params


# -------------------------
# Train Final Model
# -------------------------
model = LinearSVC(
    C=best_params["C"],
    loss=best_params["loss"],
    class_weight=weight_dict,
    max_iter=3000
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# -------------------------
# Eval
# -------------------------
print("\n=================== FINAL RESULTS ===================")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)


# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="LinearSVC_Optuna_Tuned"):

    mlflow.log_param("model", "LinearSVC")
    mlflow.log_params(best_params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("macro_recall", recall_score(y_test, y_pred, average="macro"))

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - LinearSVC")
    plt.savefig("cm_linearsvc.png")
    mlflow.log_artifact("cm_linearsvc.png")
    plt.close()

    joblib.dump(model, "linearsvc_optuna.pkl")
    mlflow.log_artifact("linearsvc_optuna.pkl")


print("\nModel Saved + Logged Successfully! 🚀🔥")


[I 2025-11-29 22:17:24,020] A new study created in memory with name: no-name-3fdff541-cf49-4f78-9bda-65f68e5c33a1


Class Weights: {np.int64(0): np.float64(0.7256444102225644), np.int64(1): np.float64(0.7350641632774342), np.int64(2): np.float64(3.8242512077294686)}


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[I 2025-11-29 22:17:40,590] Trial 0 finished with value: 0.7280048415481257 and parameters: {'C': 0.39249821903140436, 'loss': 'hinge'}. Best is trial 0 with value: 0.7280048415481257.
[I 2025-11-29 22:17:48,062] Trial 1 finished with value: 0.7219602673959397 and parameters: {'C': 0.22911616407220123, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.7280048415481257.
[I 2025-11-29 22:17:51,186] Trial 2 finished with value: 0.7149996124481258 and parameters: {'C': 0.01625036857082385, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.7280048415481257.
e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[I 2025-11-29 22:17:59,891] Trial 3 finished with val


Best Hyperparameters: {'C': 0.16302370176506276, 'loss': 'hinge'}


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



=================== FINAL RESULTS ===================
Accuracy: 0.7259498787388844

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.78      0.74      4546
           1       0.82      0.67      0.74      4488
           2       0.53      0.74      0.62       862

    accuracy                           0.73      9896
   macro avg       0.69      0.73      0.70      9896
weighted avg       0.74      0.73      0.73      9896


Confusion Matrix:
 [[3558  571  417]
 [1352 2987  149]
 [ 150   73  639]]
🏃 View run LinearSVC_Optuna_Tuned at: http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/#/experiments/7/runs/d4bc5c62ce2c4d578bbc238bf931d2b0
🧪 View experiment at: http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/#/experiments/7

Model Saved + Logged Successfully! 🚀🔥
